# Mode 1 Gmsh Polygon Workflow Demo

This notebook uses **Gmsh as the meshing backend** for a 2D polygon domain, writes the mesh to disk, reloads it, runs Mode 1 fixed-topology optimization on the reloaded mesh, and optionally launches the Gmsh viewer.

The notebook is intentionally focused on the slow path discussed earlier: arbitrary 2D polygon meshing.


In [10]:
from pathlib import Path
import json
import shutil
import sys

import jax.numpy as jnp

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from topojax import (
    initialize_mode1_domain,
    load_gmsh_msh,
    run_mode1_workflow,
    visualize_mode1_result,
)

OUTPUT_ROOT = REPO_ROOT / "outputs/topo/notebooks/mode1_domain_initializers_demo"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

GMSH_EXECUTABLE = shutil.which("gmsh") or "gmsh"
LAUNCH_GMSH_VIEWER = True
print(f"repo root: {REPO_ROOT}")
print(f"output root: {OUTPUT_ROOT}")
print(f"gmsh executable: {GMSH_EXECUTABLE}")
if shutil.which(GMSH_EXECUTABLE) is None and GMSH_EXECUTABLE == 'gmsh':
    print('WARNING: gmsh is not on PATH in this interpreter/session. The gmsh-backed cells will require gmsh to be installed.')


repo root: /home/phili/projects/TopoSmplJAX
output root: /home/phili/projects/TopoSmplJAX/outputs/topo/notebooks/mode1_domain_initializers_demo
gmsh executable: /home/phili/miniforge3/envs/jax/bin/gmsh


In [2]:
outer = jnp.asarray([
    [0.0, 0.0],
    [2.0, 0.0],
    [2.0, 1.0],
    [1.2, 1.0],
    [1.2, 2.0],
    [0.0, 2.0],
])
hole = jnp.asarray([
    [0.45, 0.45],
    [0.95, 0.45],
    [0.95, 0.95],
    [0.45, 0.95],
])

domain_kwargs = {
    "outer_boundary": outer,
    "holes": [hole],
    "target_edge_size": 0.20,
    "backend": "gmsh",
    "gmsh_executable": GMSH_EXECUTABLE,
}

domain_kwargs

{'outer_boundary': Array([[0. , 0. ],
        [2. , 0. ],
        [2. , 1. ],
        [1.2, 1. ],
        [1.2, 2. ],
        [0. , 2. ]], dtype=float32),
 'holes': [Array([[0.45, 0.45],
         [0.95, 0.45],
         [0.95, 0.95],
         [0.45, 0.95]], dtype=float32)],
 'target_edge_size': 0.2,
 'backend': 'gmsh',
 'gmsh_executable': '/home/phili/miniforge3/envs/jax/bin/gmsh'}

## 1. Build The Initial Mesh With Gmsh

This uses `initialize_mode1_domain("polygon", backend="gmsh", ...)` so the **initial** 2D arbitrary-domain mesh comes from Gmsh instead of the native in-repo mesher.


In [3]:
print('[notebook] building initial gmsh-backed polygon mesh')
gmsh_domain = initialize_mode1_domain('polygon', **domain_kwargs)
print('[notebook] initial mesh built')
print('points shape:', gmsh_domain.points.shape)
print('elements shape:', gmsh_domain.topology.elements.shape)


[notebook] building initial gmsh-backed polygon mesh
[topojax] initializing polygon domain
[notebook] initial mesh built
points shape: (132, 2)
elements shape: (212, 3)


## 2. Export The Mesh And Metrics

This runs a short Mode 1 optimization on the Gmsh-generated mesh and writes the result to `outputs/...`.


In [4]:
print('[notebook] running workflow on gmsh-generated mesh')
initial_run = run_mode1_workflow(
    gmsh_domain,
    output_dir=OUTPUT_ROOT / 'gmsh_initial',
    prefix='gmsh_initial',
    steps=8,
    step_size=0.02,
    diagnostics_every=2,
)
print('[notebook] initial workflow complete')
initial_run.artifacts


[notebook] running workflow on gmsh-generated mesh
[topojax] mode1 optimize start: nodes=132 elements=212
[topojax] mode1 export start: /home/phili/projects/TopoSmplJAX/outputs/topo/notebooks/mode1_domain_initializers_demo/gmsh_initial
[topojax] mode1 complete: mesh=/home/phili/projects/TopoSmplJAX/outputs/topo/notebooks/mode1_domain_initializers_demo/gmsh_initial/gmsh_initial_final_mesh.msh
[notebook] initial workflow complete


{'snapshot': PosixPath('/home/phili/projects/TopoSmplJAX/outputs/topo/notebooks/mode1_domain_initializers_demo/gmsh_initial/gmsh_initial_final_snapshot.npz'),
 'topo_snapshot': PosixPath('/home/phili/projects/TopoSmplJAX/outputs/topo/notebooks/mode1_domain_initializers_demo/gmsh_initial/gmsh_initial_final_snapshot.topo.npz'),
 'metrics': PosixPath('/home/phili/projects/TopoSmplJAX/outputs/topo/notebooks/mode1_domain_initializers_demo/gmsh_initial/gmsh_initial_final_metrics.json'),
 'mesh': PosixPath('/home/phili/projects/TopoSmplJAX/outputs/topo/notebooks/mode1_domain_initializers_demo/gmsh_initial/gmsh_initial_final_mesh.msh'),
 'history': PosixPath('/home/phili/projects/TopoSmplJAX/outputs/topo/notebooks/mode1_domain_initializers_demo/gmsh_initial/gmsh_initial_history.npz'),
 'history_json': PosixPath('/home/phili/projects/TopoSmplJAX/outputs/topo/notebooks/mode1_domain_initializers_demo/gmsh_initial/gmsh_initial_history.json'),
 'history_csv': PosixPath('/home/phili/projects/TopoSmp

## 3. Reload The Written Mesh

Now the workflow starts from the written `.msh` file rather than from the original domain description.


In [5]:
print('[notebook] reloading written mesh through import-msh')
imported_domain = initialize_mode1_domain(
    'import-msh',
    path=initial_run.artifacts['mesh'],
)
print('[notebook] imported mesh ready')
print('points shape:', imported_domain.points.shape)
print('elements shape:', imported_domain.topology.elements.shape)


[notebook] reloading written mesh through import-msh
[topojax] loading imported gmsh mesh
[notebook] imported mesh ready
points shape: (132, 2)
elements shape: (212, 3)


## 4. Differentiate And Move The Reloaded Mesh

This is the actual fixed-topology Mode 1 path: the mesh has already been created, written, and reloaded; now only the node coordinates move.


In [6]:
print('[notebook] running workflow on imported mesh')
imported_run = run_mode1_workflow(
    imported_domain,
    output_dir=OUTPUT_ROOT / 'gmsh_imported',
    prefix='gmsh_imported',
    steps=20,
    step_size=0.02,
    diagnostics_every=5,
)
print('[notebook] imported workflow complete')
imported_run.artifacts


[notebook] running workflow on imported mesh
[topojax] mode1 optimize start: nodes=132 elements=212
[topojax] mode1 export start: /home/phili/projects/TopoSmplJAX/outputs/topo/notebooks/mode1_domain_initializers_demo/gmsh_imported
[topojax] mode1 complete: mesh=/home/phili/projects/TopoSmplJAX/outputs/topo/notebooks/mode1_domain_initializers_demo/gmsh_imported/gmsh_imported_final_mesh.msh
[notebook] imported workflow complete


{'snapshot': PosixPath('/home/phili/projects/TopoSmplJAX/outputs/topo/notebooks/mode1_domain_initializers_demo/gmsh_imported/gmsh_imported_final_snapshot.npz'),
 'topo_snapshot': PosixPath('/home/phili/projects/TopoSmplJAX/outputs/topo/notebooks/mode1_domain_initializers_demo/gmsh_imported/gmsh_imported_final_snapshot.topo.npz'),
 'metrics': PosixPath('/home/phili/projects/TopoSmplJAX/outputs/topo/notebooks/mode1_domain_initializers_demo/gmsh_imported/gmsh_imported_final_metrics.json'),
 'mesh': PosixPath('/home/phili/projects/TopoSmplJAX/outputs/topo/notebooks/mode1_domain_initializers_demo/gmsh_imported/gmsh_imported_final_mesh.msh'),
 'history': PosixPath('/home/phili/projects/TopoSmplJAX/outputs/topo/notebooks/mode1_domain_initializers_demo/gmsh_imported/gmsh_imported_history.npz'),
 'history_json': PosixPath('/home/phili/projects/TopoSmplJAX/outputs/topo/notebooks/mode1_domain_initializers_demo/gmsh_imported/gmsh_imported_history.json'),
 'history_csv': PosixPath('/home/phili/proj

## 5. Show Numbers

Read the exported JSON files to compare the initial and imported-then-optimized runs.


In [7]:
initial_metrics = json.loads(initial_run.artifacts['metrics'].read_text(encoding='utf-8'))
initial_history = json.loads(initial_run.artifacts['history_json'].read_text(encoding='utf-8'))
imported_metrics = json.loads(imported_run.artifacts['metrics'].read_text(encoding='utf-8'))
imported_history = json.loads(imported_run.artifacts['history_json'].read_text(encoding='utf-8'))

summary = {
    'initial_mesh_kind': load_gmsh_msh(initial_run.artifacts['mesh']).primary_element_kind,
    'initial_steps': initial_metrics['n_steps'],
    'initial_final_energy': initial_metrics['final_energy'],
    'initial_final_grad_norm': initial_metrics['final_grad_norm'],
    'imported_mesh_kind': load_gmsh_msh(imported_run.artifacts['mesh']).primary_element_kind,
    'imported_steps': imported_metrics['n_steps'],
    'imported_final_energy': imported_metrics['final_energy'],
    'imported_final_grad_norm': imported_metrics['final_grad_norm'],
    'initial_history_rows': len(initial_history),
    'imported_history_rows': len(imported_history),
}
summary


{'initial_mesh_kind': 'triangle',
 'initial_steps': 8,
 'initial_final_energy': 0.0006713091861456633,
 'initial_final_grad_norm': 0.0023653730750083923,
 'imported_mesh_kind': 'triangle',
 'imported_steps': 20,
 'imported_final_energy': 0.0006690894369967282,
 'imported_final_grad_norm': 0.0023460627999156713,
 'initial_history_rows': 5,
 'imported_history_rows': 5}

## 6. Show Mesh Files And Artifacts

These are the key written files from the notebook.


In [8]:
artifact_summary = {
    'initial_mesh': str(initial_run.artifacts['mesh']),
    'initial_metrics': str(initial_run.artifacts['metrics']),
    'initial_history_json': str(initial_run.artifacts['history_json']),
    'initial_topo_snapshot': str(initial_run.artifacts['topo_snapshot']),
    'imported_mesh': str(imported_run.artifacts['mesh']),
    'imported_metrics': str(imported_run.artifacts['metrics']),
    'imported_history_json': str(imported_run.artifacts['history_json']),
    'imported_topo_snapshot': str(imported_run.artifacts['topo_snapshot']),
}

artifact_summary_path = OUTPUT_ROOT / 'artifact_summary.json'
artifact_summary_path.write_text(json.dumps(artifact_summary, indent=2), encoding='utf-8')
artifact_summary


{'initial_mesh': '/home/phili/projects/TopoSmplJAX/outputs/topo/notebooks/mode1_domain_initializers_demo/gmsh_initial/gmsh_initial_final_mesh.msh',
 'initial_metrics': '/home/phili/projects/TopoSmplJAX/outputs/topo/notebooks/mode1_domain_initializers_demo/gmsh_initial/gmsh_initial_final_metrics.json',
 'initial_history_json': '/home/phili/projects/TopoSmplJAX/outputs/topo/notebooks/mode1_domain_initializers_demo/gmsh_initial/gmsh_initial_history.json',
 'initial_topo_snapshot': '/home/phili/projects/TopoSmplJAX/outputs/topo/notebooks/mode1_domain_initializers_demo/gmsh_initial/gmsh_initial_final_snapshot.topo.npz',
 'imported_mesh': '/home/phili/projects/TopoSmplJAX/outputs/topo/notebooks/mode1_domain_initializers_demo/gmsh_imported/gmsh_imported_final_mesh.msh',
 'imported_metrics': '/home/phili/projects/TopoSmplJAX/outputs/topo/notebooks/mode1_domain_initializers_demo/gmsh_imported/gmsh_imported_final_metrics.json',
 'imported_history_json': '/home/phili/projects/TopoSmplJAX/outputs/

## 7. Launch The Gmsh Viewer

Set `LAUNCH_GMSH_VIEWER = True` in the setup cell if you want to open Gmsh automatically. Otherwise you can run the cell manually when convenient.


In [11]:
if LAUNCH_GMSH_VIEWER:
    print('[notebook] launching gmsh viewer for imported optimized mesh')
    visualize_mode1_result(
        imported_run.result,
        backend='gmsh',
        mesh_path=imported_run.artifacts['mesh'],
        gmsh_executable=GMSH_EXECUTABLE,
        wait=False,
    )
else:
    print('Set LAUNCH_GMSH_VIEWER = True in the setup cell to open the Gmsh viewer.')


[notebook] launching gmsh viewer for imported optimized mesh
